## Example async run script to run the Metadata Reviewer

In [1]:
import os
import re
import csv
import math
import asyncio
import requests
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from ai4data.metadata.reviewer import MetadataReviewerClient

In [2]:
# Load OPENAI_API_KEY from the .env file
load_dotenv(".env")
openai_api_key = os.environ["OPENAI_API_KEY"]

# Initialize the client using OpenAI
client = MetadataReviewerClient.from_openai(model="gpt-5.4-2026-03-05", api_key=openai_api_key)

In [3]:
# Load ME_API_KEY from the .env file in the current working directory
load_dotenv()
ME_API_KEY = os.getenv("ME_API_KEY")
ME_URL = 'https://metadataeditor.worldbank.org/index.php'
ME_HEADERS = {"X-API-KEY": ME_API_KEY}

# define functions for Metadata Editor operations
def fetch_me_collection_list():
    """
    Fetch collection list from the Metadata Editor.
    """
    response = requests.get(f"{ME_URL}/api/collections", headers=ME_HEADERS)
    response.raise_for_status()
    collection_list = [f"[{collection['id']}] {collection['title']}" for collection in response.json()['collections']]
    collection_list = sorted(collection_list, key=lambda s: int(re.search(r"\[(\d+)\]", s).group(1)) if re.search(r"\[(\d+)\]", s) else float("inf"))
    return collection_list

def fetch_me_project_list(collection):
    """
    Fetch project list from the Metadata Editor.
    """
    collection_id = re.search(r"\[(\d+)\]", collection).group(1)
    search_params = []
    search_params.append(f"collection={collection_id}")   
    probe_params = search_params.copy()
    probe_params.append(f"offset=0&limit=1")
    response = requests.get(f"{ME_URL}/api/editor/?{'&'.join(probe_params)}", headers=ME_HEADERS)
    total_cases = response.json().get("total", 0)
    limit = 500
    project_list = []
    num_pages = math.ceil(total_cases / limit) if limit else 0
    for offset in tqdm(range(0, total_cases, limit), total=num_pages):
        search_more_params = search_params.copy()
        search_more_params.append(f"offset={offset}&limit={limit}")
        response = requests.get(f"{ME_URL}/api/editor/?{'&'.join(search_more_params)}", headers=ME_HEADERS)
        if response.status_code != 200:
            raise Exception(f"Something wrong with the Metadata Editor search: {response.text}")
        data = response.json()
        project_list.extend(data.get("projects", []))

    project_title_list = [f"[{project['id']}] {project['title']}" for project in project_list]
    project_title_list = sorted(project_title_list, key=lambda s: int(re.search(r"\[(\d+)\]", s).group(1)) if re.search(r"\[(\d+)\]", s) else float("inf"))
    default_value = project_title_list[0] if project_title_list else None
    return project_title_list

def fetch_me_project_metadata(project):
    """
    Fetch project metadata from the Metadata Editor.
    """
    project_id = re.search(r"\[(\d+)\]", project).group(1)
    response = requests.get(f"{ME_URL}/api/editor/{project_id}", headers=ME_HEADERS)
    if response.status_code != 200:
        raise Exception(f"Something wrong with the Metadata Editor search: {response.text}")
    metadata = response.json()['project']['metadata']
    metadata.get("series_description", {}).pop("ref_country", None)
    metadata.get("series_description", {}).pop("geographic_units", None)
    return metadata

In [4]:
async def process_one(me_collection, me_project, semaphore):
    async with semaphore:    
        # fetch metadata
        metadata_to_scan = fetch_me_project_metadata(me_project)
        try:
            del metadata_to_scan["series_description"]["idno"]
            del metadata_to_scan["series_description"]["database_id"]
            del metadata_to_scan["series_description"]["time_periods"]
            del metadata_to_scan["series_description"]["ref_country"]
            del metadata_to_scan["series_description"]["geographic_units"]
            del metadata_to_scan["series_description"]["license"]
            del metadata_to_scan["series_description"]["links"]
            del metadata_to_scan["series_description"]["api_documentation"]
            del metadata_to_scan["series_description"]["alias"]
            del metadata_to_scan["series_description"]["aliases"]
        except:
            pass

        # Build URL from [12345] pattern in me_project
        m = re.search(r"\[(\d+)\]", str(me_project))
        if not m:
            raise ValueError("Could not extract project_id from ME_project (expected [12345] pattern).")
        project_id = m.group(1)
        me_url = f"{ME_URL}/api/editor/{project_id}"

        # start agents activity
        job = await client.submit_async(metadata_to_scan)

        try:
            result = await job.wait()

            return {
                "ME_collection": me_collection,
                "ME_project": me_project,
                "ME_url": me_url,
                "detected_issues": result,
            }

        except Exception as e:
            print(f"[Warning] Failed for project: {me_project}. Reason: {e}")
            return {
                "ME_collection": me_collection,
                "ME_project": me_project,
                "ME_url": me_url,
                "detected_issues": "",
            }

In [11]:
MAX_CONCURRENT_JOBS = 10
async def run_parallel(metadata_items, output_file_name):
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_JOBS)

    fieldnames = ['ME_collection', 'ME_project', 'ME_url', 'detected_issues']

    if os.path.exists(output_file_name):
        output_df = pd.read_csv(output_file_name)
    else:
        output_df = pd.DataFrame(columns=fieldnames)

    with open(output_file_name, "a", newline="") as f:
        # skip projects that are already completed
        done = set(output_df["ME_project"].astype(str))
        todo = [(collection, project) for (collection, project) in metadata_items if str(project) not in done]
        tasks = [
            asyncio.create_task(process_one(collection, project, semaphore))
            for collection, project in todo
        ]
        writer = csv.DictWriter(f, fieldnames=fieldnames)

        done_count = 0
        for future in asyncio.as_completed(tasks):
            row = await future
            writer.writerow(row)
            f.flush()
            done_count += 1
            print(f"[{done_count}/{len(todo)}] done: {row['ME_project']}")


In [12]:
me_collection = "[6] WDI - Financial Sector"
me_project_list = fetch_me_project_list(me_collection)
metadata_items = [(me_collection, me_project) for me_project in me_project_list]

100%|██████████| 1/1 [00:00<00:00,  1.44it/s]


In [13]:
output_file_name = "WDI_Financial_Sector_detected_metadata_issues_20260420.csv"
await run_parallel(metadata_items, output_file_name)

[1/37] done: [395] Wholesale price index (2010 = 100)
[2/37] done: [394] Inflation, consumer prices (annual % growth)
[3/37] done: [393] Consumer price index (2010 = 100)
[4/37] done: [387] Monetary sector credit to private sector (% of GDP)
[5/37] done: [389] Broad money (current LCU)
[6/37] done: [391] Broad money to total reserves ratio
[7/37] done: [378] Bank liquid reserves to bank assets ratio (%)
[8/37] done: [388] Claims on private sector (annual growth as % of broad money)
[9/37] done: [390] Broad money (% of GDP)
[10/37] done: [396] Deposit interest rate (%)
[11/37] done: [392] Broad money (annual % growth)
[12/37] done: [397] Lending interest rate (%)
[13/37] done: [405] Account ownership at a financial institution or with a mobile-money-service provider, poorest 40% (% of population ages 15+)
[14/37] done: [398] Interest rate spread (lending rate minus deposit rate, %)
[15/37] done: [400] Risk premium on lending (lending rate minus treasury bill rate, %)
[16/37] done: [399]